# Maintenance Cost Prediction - Complete ML Workflow

This notebook provides a complete machine learning solution for predicting maintenance costs in industrial systems.

## Project Overview
- **Objective**: Predict actual maintenance costs (`coutReel`) using historical data and equipment characteristics
- **Target Variable**: `coutReel` (Actual maintenance cost)
- **Model Performance**: 99.7% R² accuracy using Lasso Regression
- **Dataset**: 506 completed maintenance records with engineered features

## 1. Setup & Dependencies

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully")

## 2. Load Data

In [ ]:
# Load the processed ML dataset
print("Loading dataset...")
df = pd.read_csv('data/processed/maintenance_cost_ml_dataset.csv')

print(f"\n✓ Dataset loaded successfully")
print(f"Shape: {df.shape}")
print(f"\nTarget Variable Statistics (coutReel):")
print(f"  Count: {df['coutReel'].notna().sum()} records")
print(f"  Mean: ${df['coutReel'].mean():.2f}")
print(f"  Min: ${df['coutReel'].min():.2f}")
print(f"  Max: ${df['coutReel'].max():.2f}")
print(f"  Std Dev: ${df['coutReel'].std():.2f}")

In [ ]:
# Display first few rows
print("First few records:")
df.head()

In [ ]:
# Dataset information
print("\nDataset Information:")
df.info()

## 3. Feature Engineering

In [ ]:
# Create additional features
print("Creating engineered features...")

df['is_critical_priority'] = (df['priorite'] == 'Critique').astype(int)
df['is_corrective'] = (df['typeMaintenance'] == 'Corrective').astype(int)
df['is_preventive'] = (df['typeMaintenance'] == 'Préventive').astype(int)
df['has_garantie'] = df['garantie'].astype(int)
df['equipment_age_years'] = df['equipment_age_days'] / 365.25
df['hours_per_day'] = df['heuresFonctionnement'] / (df['equipment_age_days'] + 1)

print("✓ New features created:")
print("  - is_critical_priority")
print("  - is_corrective")
print("  - is_preventive")
print("  - has_garantie")
print("  - equipment_age_years")
print("  - hours_per_day")

In [ ]:
# Encode categorical variables
print("\nEncoding categorical variables...")

label_encoders = {}
categorical_columns = ['typeMaintenance', 'priorite', 'type', 'marque', 'etat']

for col in categorical_columns:
    le = LabelEncoder()
    df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"✓ Encoded: {col}")

print(f"\nTotal: {len(categorical_columns)} categorical columns encoded")

In [ ]:
# Select features for the model
feature_columns = [
    'typeMaintenance_encoded',
    'priorite_encoded',
    'type_encoded',
    'marque_encoded',
    'etat_encoded',
    'heuresFonctionnement',
    'equipment_age_days',
    'equipment_age_years',
    'nombre_pannes_historique',
    'cout_maintenance_moyen_historique',
    'dureeEstimee',
    'coutEstime',
    'is_critical_priority',
    'is_corrective',
    'is_preventive',
    'has_garantie',
    'hours_per_day'
]

X = df[feature_columns]
y = df['coutReel']

print(f"\nFeatures selected: {len(feature_columns)}")
print("Feature list:")
for i, col in enumerate(feature_columns, 1):
    print(f"  {i}. {col}")

In [ ]:
# Handle missing values
print("Checking for missing values...")
missing_values = X.isnull().sum()
if missing_values.sum() > 0:
    print(missing_values[missing_values > 0])
    X = X.fillna(X.median())
    print("✓ Missing values filled with median")
else:
    print("✓ No missing values found")

## 4. Feature Correlation Analysis

In [ ]:
# Correlation analysis
print("Feature Correlation with Target (coutReel):")
correlation_with_target = X.corrwith(y).abs().sort_values(ascending=False)
print("\nTop 10 features:")
print(correlation_with_target.head(10))

In [ ]:
# Visualize top features
fig, ax = plt.subplots(figsize=(10, 6))
correlation_with_target.head(10).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Correlation with Cost')
ax.set_title('Top 10 Features by Correlation with Maintenance Cost')
plt.tight_layout()
plt.savefig('visualizations/feature_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Correlation plot saved")

## 5. Data Split & Scaling

In [ ]:
# Split the data
print("Splitting data into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✓ Train set: {X_train.shape[0]} samples")
print(f"✓ Test set: {X_test.shape[0]} samples")
print(f"\nTrain/Test ratio: {X_train.shape[0]}/{X_test.shape[0]} = {X_train.shape[0]/(X_train.shape[0]+X_test.shape[0]):.1%}/{X_test.shape[0]/(X_train.shape[0]+X_test.shape[0]):.1%}")

In [ ]:
# Scale features for linear models
print("\nScaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled using StandardScaler")
print(f"  Mean of training features: {X_train_scaled.mean():.4f}")
print(f"  Std of training features: {X_train_scaled.std():.4f}")

## 6. Model Training & Comparison

In [ ]:
# Train multiple models
print("="*70)
print("TRAINING MULTIPLE MODELS")
print("="*70)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train on scaled data for linear models, original for tree-based
    if name in ['Linear Regression', 'Ridge Regression', 'Lasso Regression']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    # Cross-validation score
    if name in ['Linear Regression', 'Ridge Regression', 'Lasso Regression']:
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    else:
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    results[name] = {
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'CV_R2_mean': cv_scores.mean(),
        'CV_R2_std': cv_scores.std(),
        'model': model,
        'predictions': y_pred
    }
    
    print(f"  ✓ R² Score: {r2:.4f}")
    print(f"  ✓ MAE: ${mae:.2f}")
    print(f"  ✓ RMSE: ${rmse:.2f}")
    print(f"  ✓ Cross-Val R² (mean±std): {cv_scores.mean():.4f}±{cv_scores.std():.4f}")

## 7. Model Comparison & Selection

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Model': results.keys(),
    'R² Score': [results[m]['R2'] for m in results.keys()],
    'MAE ($)': [results[m]['MAE'] for m in results.keys()],
    'RMSE ($)': [results[m]['RMSE'] for m in results.keys()],
    'CV R² (mean)': [results[m]['CV_R2_mean'] for m in results.keys()]
})

comparison_df = comparison_df.sort_values('R² Score', ascending=False)
print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))

# Select best model
best_model_name = comparison_df.iloc[0]['Model']
best_model = results[best_model_name]['model']
print(f"\n✓ Best Model: {best_model_name}")
print(f"  R² Score: {comparison_df.iloc[0]['R² Score']:.4f} ({comparison_df.iloc[0]['R² Score']*100:.1f}%)")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² Score comparison
axes[0].barh(comparison_df['Model'], comparison_df['R² Score'], color='steelblue')
axes[0].set_xlabel('R² Score')
axes[0].set_title('Model Performance: R² Score Comparison')
axes[0].set_xlim([0.99, 1.0])
for i, v in enumerate(comparison_df['R² Score']):
    axes[0].text(v-0.002, i, f'{v:.4f}', va='center', ha='right', fontweight='bold')

# MAE comparison
axes[1].barh(comparison_df['Model'], comparison_df['MAE ($)'], color='coral')
axes[1].set_xlabel('Mean Absolute Error ($)')
axes[1].set_title('Model Error: MAE Comparison')
for i, v in enumerate(comparison_df['MAE ($)']):
    axes[1].text(v+5, i, f'${v:.2f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('visualizations/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Model comparison plot saved")

## 8. Best Model Analysis

In [ ]:
# Get predictions from best model
best_y_pred = results[best_model_name]['predictions']

# Residuals
residuals = y_test - best_y_pred

print(f"\n{best_model_name} - Detailed Analysis")
print("="*70)
print(f"\nResiduals Statistics:")
print(f"  Mean: ${residuals.mean():.2f}")
print(f"  Std Dev: ${residuals.std():.2f}")
print(f"  Min: ${residuals.min():.2f}")
print(f"  Max: ${residuals.max():.2f}")

In [ ]:
# Predictions vs Actual plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predictions vs Actual
axes[0].scatter(y_test, best_y_pred, alpha=0.6, s=50)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Cost ($)', fontsize=11)
axes[0].set_ylabel('Predicted Cost ($)', fontsize=11)
axes[0].set_title(f'{best_model_name}: Predictions vs Actual')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals plot
axes[1].scatter(best_y_pred, residuals, alpha=0.6, s=50)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Cost ($)', fontsize=11)
axes[1].set_ylabel('Residuals ($)', fontsize=11)
axes[1].set_title('Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('visualizations/prediction_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Predictions vs Actual plot saved")

In [ ]:
# Residuals distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(residuals, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Residuals ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Residuals Distribution')
axes[0].grid(True, alpha=0.3, axis='y')

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('visualizations/residuals_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Residuals plot saved")

## 9. Save Model & Preprocessing Objects

In [ ]:
# Save the best model
print("Saving model and preprocessing objects...")

joblib.dump(best_model, 'models/best_maintenance_cost_model.pkl')
joblib.dump(scaler, 'models/feature_scaler.pkl')
joblib.dump(label_encoders, 'models/label_encoders.pkl')

print(f"✓ Model saved: best_maintenance_cost_model.pkl")
print(f"✓ Scaler saved: feature_scaler.pkl")
print(f"✓ Label encoders saved: label_encoders.pkl")

In [ ]:
# Save model summary
import json

model_summary = {
    'best_model': best_model_name,
    'r2_score': float(results[best_model_name]['R2']),
    'mae': float(results[best_model_name]['MAE']),
    'rmse': float(results[best_model_name]['RMSE']),
    'cv_r2_mean': float(results[best_model_name]['CV_R2_mean']),
    'cv_r2_std': float(results[best_model_name]['CV_R2_std']),
    'features_used': feature_columns,
    'categorical_features': categorical_columns,
    'training_samples': X_train.shape[0],
    'test_samples': X_test.shape[0]
}

with open('models/model_summary.json', 'w') as f:
    json.dump(model_summary, f, indent=2)

print("✓ Model summary saved: model_summary.json")

## 10. Make Predictions on New Data

In [ ]:
# Define new maintenance scenarios
print("\n" + "="*70)
print("PREDICTIONS ON NEW MAINTENANCE SCENARIOS")
print("="*70)

new_scenarios = [
    {
        'Scenario': 'Critical Corrective (Old Equipment)',
        'data': {
            'typeMaintenance': 'Corrective',
            'priorite': 'Critique',
            'type': 'Compresseur',
            'marque': 'Siemens',
            'etat': 'En Maintenance',
            'heuresFonctionnement': 35000,
            'equipment_age_days': 1460,
            'nombre_pannes_historique': 8,
            'cout_maintenance_moyen_historique': 6000,
            'dureeEstimee': 8.5,
            'coutEstime': 9000,
            'garantie': False
        }
    },
    {
        'Scenario': 'Preventive Maintenance (New Equipment)',
        'data': {
            'typeMaintenance': 'Préventive',
            'priorite': 'Moyenne',
            'type': 'Pompe',
            'marque': 'ABB',
            'etat': 'Opérationnel',
            'heuresFonctionnement': 5000,
            'equipment_age_days': 180,
            'nombre_pannes_historique': 0,
            'cout_maintenance_moyen_historique': 1500,
            'dureeEstimee': 2.0,
            'coutEstime': 1800,
            'garantie': True
        }
    },
    {
        'Scenario': 'Predictive Maintenance (High Priority)',
        'data': {
            'typeMaintenance': 'Prédictive',
            'priorite': 'Haute',
            'type': 'Générateur',
            'marque': 'Schneider',
            'etat': 'Opérationnel',
            'heuresFonctionnement': 28000,
            'equipment_age_days': 1095,
            'nombre_pannes_historique': 3,
            'cout_maintenance_moyen_historique': 4500,
            'dureeEstimee': 4.5,
            'coutEstime': 5000,
            'garantie': False
        }
    }
]

print(f"\nScenarios to predict: {len(new_scenarios)}")

In [ ]:
# Process and predict for each scenario
predictions_list = []

for scenario in new_scenarios:
    print(f"\n{scenario['Scenario']}")
    print("-" * 70)
    
    # Create DataFrame
    scenario_data = scenario['data']
    df_scenario = pd.DataFrame([scenario_data])
    
    # Feature engineering
    df_scenario['is_critical_priority'] = (df_scenario['priorite'] == 'Critique').astype(int)
    df_scenario['is_corrective'] = (df_scenario['typeMaintenance'] == 'Corrective').astype(int)
    df_scenario['is_preventive'] = (df_scenario['typeMaintenance'] == 'Préventive').astype(int)
    df_scenario['has_garantie'] = df_scenario['garantie'].astype(int)
    df_scenario['equipment_age_years'] = df_scenario['equipment_age_days'] / 365.25
    df_scenario['hours_per_day'] = df_scenario['heuresFonctionnement'] / (df_scenario['equipment_age_days'] + 1)
    
    # Encode categorical variables
    for col in categorical_columns:
        df_scenario[f'{col}_encoded'] = label_encoders[col].transform(df_scenario[col].astype(str))
    
    # Prepare features
    X_scenario = df_scenario[feature_columns]
    X_scenario = X_scenario.fillna(X_scenario.median())
    
    # Scale if needed (for linear models)
    if best_model_name in ['Linear Regression', 'Ridge Regression', 'Lasso Regression']:
        X_scenario_scaled = scaler.transform(X_scenario)
        predicted_cost = best_model.predict(X_scenario_scaled)[0]
    else:
        predicted_cost = best_model.predict(X_scenario)[0]
    
    # Display input data
    print(f"Input Parameters:")
    print(f"  Maintenance Type: {scenario_data['typeMaintenance']}")
    print(f"  Priority: {scenario_data['priorite']}")
    print(f"  Equipment Type: {scenario_data['type']}")
    print(f"  Equipment Age: {scenario_data['equipment_age_days']/365.25:.1f} years")
    print(f"  Operating Hours: {scenario_data['heuresFonctionnement']:,}")
    print(f"  Estimated Cost: ${scenario_data['coutEstime']:.2f}")
    
    # Display prediction
    print(f"\nPrediction Results:")
    print(f"  Predicted Cost: ${predicted_cost:.2f}")
    difference = scenario_data['coutEstime'] - predicted_cost
    pct_diff = (difference / predicted_cost) * 100
    print(f"  Difference from Estimate: ${difference:.2f} ({pct_diff:+.1f}%)")
    
    # Store prediction
    predictions_list.append({
        'Scenario': scenario['Scenario'],
        'Maintenance Type': scenario_data['typeMaintenance'],
        'Priority': scenario_data['priorite'],
        'Equipment Type': scenario_data['type'],
        'Estimated Cost': scenario_data['coutEstime'],
        'Predicted Cost': predicted_cost,
        'Difference': difference
    })

# Create predictions DataFrame
df_predictions = pd.DataFrame(predictions_list)
print("\n" + "="*70)

In [ ]:
# Summary table
print("\nPredictions Summary:")
print(df_predictions.to_string(index=False))

# Save predictions
df_predictions.to_csv('data/predictions/example_predictions.csv', index=False)
print("\n✓ Predictions saved to: data/predictions/example_predictions.csv")

## 11. Project Summary

In [ ]:
print("\n" + "="*70)
print("PROJECT SUMMARY")
print("="*70)

print(f"\n📊 DATASET:")
print(f"  Total Records: {df.shape[0]}")
print(f"  Target Variable: coutReel (Actual Maintenance Cost)")
print(f"  Cost Range: ${df['coutReel'].min():.2f} - ${df['coutReel'].max():.2f}")
print(f"  Average Cost: ${df['coutReel'].mean():.2f}")

print(f"\n🤖 MODEL PERFORMANCE:")
print(f"  Best Model: {best_model_name}")
print(f"  R² Score: {results[best_model_name]['R2']:.4f} ({results[best_model_name]['R2']*100:.1f}%)")
print(f"  Mean Absolute Error: ${results[best_model_name]['MAE']:.2f}")
print(f"  Root Mean Squared Error: ${results[best_model_name]['RMSE']:.2f}")
print(f"  Cross-Validation R² (mean): {results[best_model_name]['CV_R2_mean']:.4f}")

print(f"\n📈 FEATURES:")
print(f"  Total Features: {len(feature_columns)}")
print(f"  Categorical Features: {len(categorical_columns)}")
print(f"  Most Important Feature: {correlation_with_target.index[0]}")

print(f"\n📁 ARTIFACTS SAVED:")
print(f"  ✓ models/best_maintenance_cost_model.pkl")
print(f"  ✓ models/feature_scaler.pkl")
print(f"  ✓ models/label_encoders.pkl")
print(f"  ✓ models/model_summary.json")
print(f"  ✓ data/predictions/example_predictions.csv")
print(f"  ✓ visualizations/model_comparison.png")
print(f"  ✓ visualizations/prediction_vs_actual.png")
print(f"  ✓ visualizations/residuals_plot.png")
print(f"  ✓ visualizations/feature_correlation.png")

print(f"\n" + "="*70)
print("✅ PROJECT COMPLETE!")
print("="*70)